![Week 3 Day 3](../assets/banners/w3d3.png)

# Week 3 Day 3 - LangGraph: State, Nodes, Edges and Your First Graph

Yesterday ended with a reveal: the agent `create_agent` built for you was a LangGraph graph all along, with a `model` node, a `tools` node and a conditional edge looping between them. Today you take the pen. 🖊️

You'll learn LangGraph's five core ideas (**state, nodes, edges, reducers and supersteps**) and then build your first `StateGraph`: Emma's *rubric checker*, which routes questions by type and **loops** until the marks add up. It's a small version of the validate → retry loop you'll ship on Friday.

And here's the best part: **today needs no API key at all.** A LangGraph node is just a Python function. It can call an LLM, but it doesn't have to, so we'll learn graphs with plain Python and keep every cell free, fast and repeatable.

![Your 6-week roadmap](../assets/roadmap/week3.png)

### 🎯 By the end of this lab you will
1. 🧠 Explain state, nodes, edges, reducers and supersteps, with a diagram for each
2. 🧮 Use a **reducer** (`Annotated[list, operator.add]`) so parallel nodes can write to the same key
3. 🔀 Route with **conditional edges**
4. 🔁 Build a **loop** with a retry cap, and know what `GraphRecursionError` means
5. 🕸️ Draw any graph with `graph.get_graph().draw_mermaid()`

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 3 - LangGraph Concepts: State, Nodes, Edges, Reducers, Supersteps | 🎬 |
| 2 | Day 3 - Your First StateGraph with Conditional Routing and Loops | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything in this notebook runs <b>offline</b>. No 🔑 cells today! When you see a node doing keyword matching or simple arithmetic, picture an LLM call in its place: LangGraph doesn't care what happens <i>inside</i> a node, only what it <b>returns</b>.</span>
        </td>
    </tr>
</table>

## Imports

In [ ]:
import copy
import operator
from typing import Annotated, Literal, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph

from gradewise.config import DATA_DIR
from gradewise.schemas import ExamPaper

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in exam.questions}
print(f"✅ LangGraph ready. Loaded {len(QUESTIONS)} questions from {exam.exam_id}.")

## Part 1: The five ideas

| Idea | What it is | In code |
|---|---|---|
| **State** | The shared "whiteboard" every node reads from and writes to | a `TypedDict` |
| **Node** | A step: reads the state, returns a **partial update** (only the keys it changed) | a Python function |
| **Edge** | Which node runs next: fixed, or decided by a function | `add_edge`, `add_conditional_edges` |
| **Reducer** | *How* an update is merged into the state: overwrite (default) or combine | `Annotated[list, operator.add]` |
| **Superstep** | One "tick": every node that's ready runs (in parallel), then all updates are applied together | automatic |

![Diagram](../assets/diagrams/23e9c1682c5a.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([START]) --> A[node A] --> B{router}
  B -- "route 1" --> C[node C]
  B -- "route 2" --> D[node D]
  C --> E([END])
  D --> A
```

</details>


🏫 **An analogy.** The state is the whiteboard in Emma's staff room. Each marker (node) walks in, reads the board, and writes *only* their own results. The reducer is the house rule for writing: "replace what's there" or "add to the list". A superstep is one round in which everyone who's ready works at the same time.

### Your first graph: two nodes
The recipe never changes: **define the state → write the nodes → wire the edges → compile → invoke.**

In [ ]:
class AnswerState(TypedDict):
    answer: str
    word_count: int
    verdict: str


def count_words(state: AnswerState) -> dict:
    return {"word_count": len(state["answer"].split())}  # 💡 a PARTIAL update: just one key


def judge_length(state: AnswerState) -> dict:
    return {"verdict": "too short to earn 4 marks" if state["word_count"] < 15 else "long enough"}


builder = StateGraph(AnswerState)
builder.add_node("count_words", count_words)
builder.add_node("judge_length", judge_length)
builder.add_edge(START, "count_words")
builder.add_edge("count_words", "judge_length")
builder.add_edge("judge_length", END)
graph = builder.compile()

graph.invoke({"answer": "Deadlock is when processes wait for each other forever."})

In [ ]:
from gradewise.display import show_graph

show_graph(graph)  # 🧪 TRY: paste the output into https://mermaid.live

### Reducers: what happens when two updates meet
By default, a node's update **overwrites** the key. That's right for `word_count`, but wrong for a list of comments that several nodes contribute to. A **reducer** is a function `(old, new) → merged`, attached with `Annotated`:

In [ ]:
class FeedbackState(TypedDict):
    latest: str  # no reducer: overwrite
    comments: Annotated[list[str], operator.add]  # reducer: old + new


def spelling(state: FeedbackState) -> dict:
    return {"latest": "spelling", "comments": ["✏️ 'semafore' → 'semaphore' (no marks lost)"]}


def content(state: FeedbackState) -> dict:
    return {"latest": "content", "comments": ["🧠 Missing: the difference between binary and counting semaphores"]}


builder = StateGraph(FeedbackState)
builder.add_node("spelling", spelling)
builder.add_node("content", content)
builder.add_edge(START, "spelling")
builder.add_edge("spelling", "content")
builder.add_edge("content", END)

builder.compile().invoke({"latest": "", "comments": ["📄 Q4b received"]})

💡 `latest` was overwritten (only `"content"` is left), while `comments` **accumulated**, including the one we passed in. For chat messages, LangGraph ships a smarter reducer, `add_messages`, which also updates messages by ID: that's what `create_agent` uses for its `messages` key.

### Supersteps: parallel nodes, applied together
If two edges leave `START`, both nodes are ready at the same time, so they run in the **same superstep**. Their updates are applied together at the end of it. Let's prove it: each node can read which step it's in from its `config`.

In [ ]:
def check_keywords(state: FeedbackState, config: RunnableConfig) -> dict:
    print(f"check_keywords ran in superstep {config['metadata']['langgraph_step']}")
    return {"comments": ["🔑 mentions wait() and signal()"]}


def check_length(state: FeedbackState, config: RunnableConfig) -> dict:
    print(f"check_length   ran in superstep {config['metadata']['langgraph_step']}")
    return {"comments": ["📏 about 80 words: fine"]}


def summarise(state: FeedbackState, config: RunnableConfig) -> dict:
    step, seen = config["metadata"]["langgraph_step"], len(state["comments"])
    print(f"summarise      ran in superstep {step} and sees {seen} comments")
    return {"latest": "summary"}


builder = StateGraph(FeedbackState)
for fn in (check_keywords, check_length, summarise):
    builder.add_node(fn.__name__, fn)
builder.add_edge(START, "check_keywords")
builder.add_edge(START, "check_length")  # 💡 fan-out: two edges from START
builder.add_edge(["check_keywords", "check_length"], "summarise")  # 💡 fan-in: wait for BOTH
builder.add_edge("summarise", END)
parallel = builder.compile()

parallel.invoke({"latest": "", "comments": []})

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Predict: what happens if we remove the reducer, so <code>comments</code> is a plain <code>list[str]</code>, and two nodes write to it <b>in the same superstep</b>? Overwrite with the last one? Keep the first? Something else? Run the next cell to find out.</span>
        </td>
    </tr>
</table>

In [ ]:
class NoReducer(TypedDict):
    latest: str
    comments: list[str]  # ⚠️ no reducer!


builder = StateGraph(NoReducer)
builder.add_node("check_keywords", lambda s: {"comments": ["🔑 keywords ok"]})
builder.add_node("check_length", lambda s: {"comments": ["📏 length ok"]})
builder.add_edge(START, "check_keywords")
builder.add_edge(START, "check_length")

try:
    builder.compile().invoke({"latest": "", "comments": []})
except InvalidUpdateError as err:
    print("💥 InvalidUpdateError:", str(err).splitlines()[0])

LangGraph refuses to guess. Two writes to a plain key in one superstep is an error, not a silent overwrite, and the fix is in the message: *use an Annotated key*, i.e. a reducer. You'll need exactly this on day 4, when 5 markers run in parallel.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Three classic LangGraph mistakes: <b>(1)</b> returning the <i>whole</i> state from a node instead of just the keys you changed (with a reducer, that duplicates your list!); <b>(2)</b> mutating <code>state</code> in place (<code>state["comments"].append(...)</code>) instead of returning an update: the change may silently vanish; <b>(3)</b> forgetting the reducer on a key that parallel nodes write to.</span>
        </td>
    </tr>
</table>

## Part 2: Conditional routing

A **conditional edge** is a function that looks at the state and returns the **name of the next node**. Let's route each question to the right kind of checker: numerical questions need the *working* checked; written ones need key terms.

💡 Pass the list of possible destinations as the third argument. LangGraph uses it to validate your graph and to draw it correctly.

In [ ]:
from gradewise.display import show_graph

class RouteState(TypedDict):
    question_id: str
    checks: Annotated[list[str], operator.add]


def classify(state: RouteState) -> dict:
    q = QUESTIONS[state["question_id"]]
    return {"checks": [f"{q.id} is a {q.question_type} question worth {q.max_marks:g}"]}


def numeric_rules(state: RouteState) -> dict:
    return {"checks": ["🔢 award method marks for correct working, even if the final number is wrong"]}


def written_rules(state: RouteState) -> dict:
    return {"checks": ["📝 look for the key terms; ignore spelling"]}


def pick_rules(state: RouteState) -> Literal["numeric_rules", "written_rules"]:
    return "numeric_rules" if QUESTIONS[state["question_id"]].question_type == "numerical" else "written_rules"


builder = StateGraph(RouteState)
for fn in (classify, numeric_rules, written_rules):
    builder.add_node(fn.__name__, fn)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", pick_rules, ["numeric_rules", "written_rules"])
builder.add_edge("numeric_rules", END)
builder.add_edge("written_rules", END)
router = builder.compile()

for qid in ["Q5", "Q3"]:
    print(router.invoke({"question_id": qid, "checks": []})["checks"])
show_graph(router)

## Part 3: Loops: Emma's rubric checker 🔁

Now the main event. In week 1 you saw that an LLM sometimes extracts a question whose criteria **don't add up** to its marks. Today's graph checks a question, and if something's wrong, sends it to a `repair` node and checks again, until it's clean or we run out of tries:

![Diagram](../assets/diagrams/9b152ef66fcf.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([START]) --> C{check}
  C -- "problems, tries left" --> R[repair] --> C
  C -- clean --> A[approve] --> E([END])
  C -- "out of tries" --> X[escalate to Emma] --> E
```

</details>


First, a deliberately broken copy of Q4b (semaphores): one criterion is worth too much and the methodology is missing.

In [ ]:
broken = copy.deepcopy(QUESTIONS["Q4b"].model_dump())
broken["criteria"][3]["marks"] = 2.0  # ⚠️ now the criteria add up to 5, but Q4b is worth 4
broken["methodology"] = ""  # ⚠️ and the examiner's instructions are gone

print(
    f"{broken['id']}: max {broken['max_marks']:g}, criteria add up to {sum(c['marks'] for c in broken['criteria']):g}"
)

In [ ]:
class CheckState(TypedDict):
    question: dict  # 💡 a plain dict, so any checkpointer can save it (day 4)
    problems: list[str]
    attempts: int
    max_attempts: int
    log: Annotated[list[str], operator.add]


def check(state: CheckState) -> dict:
    q = state["question"]
    total = round(sum(c["marks"] for c in q["criteria"]), 2)
    problems = []
    if total != q["max_marks"]:
        problems.append(f"criteria add up to {total:g}, but the question is worth {q['max_marks']:g}")
    if not q["methodology"].strip():
        problems.append("no marking methodology")
    return {"problems": problems, "log": [f"check   → {problems or 'clean ✅'}"]}


def repair(state: CheckState) -> dict:
    # 🎭 A stand-in for an LLM retry: it fixes ONE problem per attempt. On Friday this node
    #    becomes a real model call that sees its previous answer plus the list of problems.
    q = copy.deepcopy(state["question"])  # ⚠️ never mutate the state in place
    problem = state["problems"][0]
    if problem.startswith("criteria"):
        excess = sum(c["marks"] for c in q["criteria"]) - q["max_marks"]
        q["criteria"][-1]["marks"] -= excess
    else:
        q["methodology"] = "Award each criterion independently. Ignore spelling."
    attempt = state["attempts"] + 1
    return {"question": q, "attempts": attempt, "log": [f"repair  → attempt {attempt}: fixed '{problem}'"]}


def approve(state: CheckState) -> dict:
    return {"log": ["approve → ready for grading 🎉"]}


def escalate(state: CheckState) -> dict:
    return {"log": [f"escalate → Emma, please fix by hand: {state['problems']}"]}


def after_check(state: CheckState) -> Literal["approve", "repair", "escalate"]:
    if not state["problems"]:
        return "approve"
    if state["attempts"] < state["max_attempts"]:
        return "repair"
    return "escalate"  # 💡 out of tries: a human decides

In [ ]:
from gradewise.display import show_graph

builder = StateGraph(CheckState)
for fn in (check, repair, approve, escalate):
    builder.add_node(fn.__name__, fn)
builder.add_edge(START, "check")
builder.add_conditional_edges("check", after_check, ["approve", "repair", "escalate"])
builder.add_edge("repair", "check")  # 🔁 the loop
builder.add_edge("approve", END)
builder.add_edge("escalate", END)
checker = builder.compile()

show_graph(checker)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Our question has <b>two</b> problems and <code>repair</code> fixes one per attempt. With <code>max_attempts=3</code>, how many times will <code>check</code> run? And with <code>max_attempts=1</code>, which node ends the run?</span>
        </td>
    </tr>
</table>

In [ ]:
result = checker.invoke({"question": broken, "problems": [], "attempts": 0, "max_attempts": 3, "log": []})
print("\n".join(result["log"]))

In [ ]:
result = checker.invoke({"question": broken, "problems": [], "attempts": 0, "max_attempts": 1, "log": []})
print("\n".join(result["log"]))

### Watching a graph run with `stream`
`invoke` gives you only the final state. `stream(..., stream_mode="updates")` yields each node's update **as it happens**: perfect for progress bars and for debugging loops. (Day 4 covers all the stream modes.)

In [ ]:
for update in checker.stream(
    {"question": broken, "problems": [], "attempts": 0, "max_attempts": 3, "log": []}, stream_mode="updates"
):
    for node, change in update.items():
        print(f"{node:8} changed {sorted(change)}")

### The safety net: `recursion_limit`
What if `after_check` had a bug and never returned `"approve"`? LangGraph counts supersteps, and stops at the **recursion limit** with a `GraphRecursionError`. ⚠️ In the LangGraph version installed for this course the default limit is **10,007** supersteps (older versions used 25), so with an LLM in the loop you'd pay for thousands of calls before it trips. Set it yourself, per run, in the config. Let's set it low to see it:

In [ ]:
def buggy_router(state: CheckState) -> str:
    return "repair"  # ⚠️ the bug: never stops


builder = StateGraph(CheckState)
builder.add_node("check", check)
builder.add_node("repair", lambda s: {"attempts": s["attempts"] + 1})
builder.add_edge(START, "check")
builder.add_conditional_edges("check", buggy_router, ["repair"])
builder.add_edge("repair", "check")

try:
    builder.compile().invoke(
        {"question": broken, "problems": [], "attempts": 0, "max_attempts": 3, "log": []}, {"recursion_limit": 8}
    )
except GraphRecursionError as err:
    print("🛑 GraphRecursionError:", str(err).splitlines()[0])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">The recursion limit is a <b>safety net, not a design</b>. Always put your own cap in the state (<code>attempts</code> / <code>max_attempts</code>) and route to a graceful ending, like <code>escalate</code>. With a real LLM in the loop, every lap costs money, and a <code>GraphRecursionError</code> throws away everything the run produced.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Notice what the graph gives Emma that a single prompt never could: a <b>log of every decision</b>, a <b>guaranteed upper bound</b> on attempts (and so on cost), and a <b>defined path to a human</b> when automation fails. That's what makes AI acceptable in high-stakes settings like exams, medicine or finance: not perfection, but predictable behaviour when things go wrong.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Open <code>gradewise/exam/graph.py</code> and find <code>after_validate</code>: it's today's <code>after_check</code>, almost line for line. <code>validate</code> calls <code>ExamPaper.problems()</code> (in <code>gradewise/schemas/exam.py</code>) instead of our hand-written checks, and <code>retry</code> sends the problems back to a real model. On Friday you'll run it.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🧠 learned LangGraph's five ideas: **state, nodes, edges, reducers, supersteps**
- 🧮 used `Annotated[list, operator.add]`, and saw `InvalidUpdateError` when parallel writes have no reducer
- 🔀 routed questions by type with `add_conditional_edges`
- 🔁 built Emma's rubric checker: a **check → repair** loop with a retry cap and an escape hatch to a human
- 🛑 met the `recursion_limit` safety net, and why it shouldn't be your only one

**Tomorrow (Day 4):** the power tools. `Send` to mark 5 answers in parallel, checkpoints and **time travel**, `interrupt()` to pause for Emma, subgraphs, and every streaming mode.

⏭ **Next:** `week3/day4.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Extend the rubric checker. <b>(1)</b> Add a rule to <code>check</code>: every criterion must have at least one keyword. Make <code>repair</code> fix it by taking the longest two words from the criterion's description. <b>(2)</b> Add a <code>second_marker</code> node: questions worth <b>5 marks or more</b> must visit it after they're clean (it logs "👥 needs a second marker") before <code>approve</code>. Test it on a broken Q5 (worth 5) and a broken Q3 (worth 4), and print both graphs with <code>draw_mermaid()</code>. Solution: <code>week3/solutions/day3_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangGraph overview: <a href="https://docs.langchain.com/oss/python/langgraph/overview">docs.langchain.com/oss/python/langgraph/overview</a><br/> • The Graph API (state, reducers, edges): <a href="https://docs.langchain.com/oss/python/langgraph/graph-api">docs.langchain.com/oss/python/langgraph/graph-api</a><br/> • Thinking in LangGraph: <a href="https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph">docs.langchain.com/oss/python/langgraph/thinking-in-langgraph</a><br/> • Errors explained (INVALID_CONCURRENT_GRAPH_UPDATE, GRAPH_RECURSION_LIMIT): <a href="https://docs.langchain.com/oss/python/langgraph/errors">docs.langchain.com/oss/python/langgraph/errors</a></span>
        </td>
    </tr>
</table>